# Massachusetts Roads: a second dataset

**Inputs:** the dataset `balraj98/massachusetts-roads-dataset` (the v4 model comes with the repository).
Settings: GPU T4, Internet on. Save Version, Save & Run All.

* **Part A** (about 10 minutes): the v4 model, trained only on DeepGlobe, applied to the 49 official
  test tiles without retraining. Massachusetts has 1 m pixels and v4 was trained on 0.5 m pixels, so
  it is run twice: on the tiles as they are, and on tiles enlarged two times to 0.5 m pixels.
* **Part B** (several hours): our network trained on the 1,108 official training tiles with the
  v4 recipe, best epoch chosen on the 14 official validation tiles, then scored on the 49 test tiles.
  If it ends with `RESUME NEEDED`, attach this notebook's output and run a new version.

The official split is used by most published Massachusetts results, so part B can be compared
with them directly. Output: `/kaggle/working/massachusetts_results.zip`.

In [ ]:
# Settings
RUN_ZERO_SHOT = True
TRAIN_EXPERIMENT = "mass_scratch"   # or "mass_finetune" (start from the v4 weights)
RUN_TRAINING = True
SMOKE_TEST = False
TIME_LIMIT_H = 11.0
GSD_SCALE = 0.5   # 0.5 m / 1 m: gap bridging distances halve in pixels
VARIANTS = {"single": ["raw"], "tta": ["raw", "full"]}
OUT = "/kaggle/working/massachusetts_results"

In [ ]:
# Code from the repository
import os, sys, glob, json, subprocess, shutil, zipfile
REPO_URL = "https://github.com/Dilraj07/Satellite-Imagery-based-Road-Network-Extraction.git"
REPO_REF = "main"          # a branch, or a commit hash to pin the exact code
REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO):
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO]).returncode:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)          # REPO_REF is a commit
        subprocess.run(["git", "-C", REPO, "checkout", "-q", REPO_REF], check=True)
V4 = os.path.join(REPO, "models", "best_model_v4.pth")   # the paper's model, included in the repository
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
try:
    import skimage
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image"], check=True)
sys.path.insert(0, os.path.join(REPO, "scripts", "experiments"))
import numpy as np, pandas as pd, torch
import exp_data as D
import exp_torch as T
DEVICE = torch.device("cuda")
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
def table(rows):
    """rows: [(name, params, summary dict)] -> printed table and LaTeX lines."""
    keys = [("iou", "IoU"), ("f1", "F1"), ("relaxed_f1", "Rel. F1"), ("cldice", "clDice"), ("canopy_recall", "Canopy R")]
    df = pd.DataFrame([{"model": n, "params (M)": round(p / 1e6, 2) if p else None,
                        **{h: round(100 * s[k], 1) for k, h in keys},
                        "parts": s["median_components_per_tile"]} for n, p, s in rows])
    print(df.to_string(index=False))
    latex = [f"{n} & {p / 1e6:.2f} & " + " & ".join(f"{100 * s[k]:.1f}" for k, _ in keys) + " \\\\" for n, p, s in rows]
    return df, latex

In [ ]:
# Dataset
os.makedirs(OUT, exist_ok=True)
ROOT = D.find_massachusetts()
split = D.massachusetts_split(ROOT)
print(ROOT, {k: len(v) for k, v in split.items()}, "(official: train 1108, select/val 14, test 49)")
print("image size", D.read_rgb(split["test"][0][1]).shape, "| label pixels", int(D.read_mask(split["test"][0][2]).sum()))
assert split["train"] and split["test"] and split["select"]
VEG_T = D.otsu_vegetation_threshold(split["test"])
RESULTS = {"vegetation_threshold": VEG_T, "n": {k: len(v) for k, v in split.items()}}

In [ ]:
# Part A: v4 without retraining
if RUN_ZERO_SHOT:
    model, _ = T.load_any_checkpoint(V4, DEVICE)
    rows = []
    for scale in (1.0, 2.0):
        rec = T.evaluate(model, split["test"], VARIANTS, VEG_T, DEVICE, scale=scale, gsd_scale=GSD_SCALE)
        RESULTS[f"zero_shot_scale{scale:g}"] = {v: D.summary(rec, v) for v in ("single_raw", "tta_raw", "tta_full")}
        rows += [(f"v4, scale {scale:g}, {v}", T.n_params(model), RESULTS[f"zero_shot_scale{scale:g}"][v])
                 for v in ("tta_raw", "tta_full")]
    del model; torch.cuda.empty_cache()
    table(rows)
else:
    print("part A skipped")

In [ ]:
# Part B: train on Massachusetts
TRAIN_OUT = f"/kaggle/working/{TRAIN_EXPERIMENT}" + ("_smoke" if SMOKE_TEST else "")
status = None
if RUN_TRAINING:
    init = V4 if T.EXPERIMENTS[TRAIN_EXPERIMENT].get("init") == "v4" else None
    status = T.train_experiment(TRAIN_EXPERIMENT, split["train"], split["select"], TRAIN_OUT,
                                init_checkpoint=init, time_limit_h=TIME_LIMIT_H, smoke_test=SMOKE_TEST)

In [ ]:
# Part B: test
best = f"{TRAIN_OUT}/best_{TRAIN_EXPERIMENT}.pth"
if status == "DONE" and os.path.exists(best):
    model, meta = T.load_any_checkpoint(best, DEVICE)
    rec = T.evaluate(model, split["test"], VARIANTS, VEG_T, DEVICE, gsd_scale=GSD_SCALE)
    RESULTS[TRAIN_EXPERIMENT] = {v: D.summary(rec, v) for v in ("single_raw", "tta_raw", "tta_full")}
    RESULTS[TRAIN_EXPERIMENT]["epoch"] = meta.get("epoch")
    table([(f"{TRAIN_EXPERIMENT}, {v}", T.n_params(model), RESULTS[TRAIN_EXPERIMENT][v])
           for v in ("single_raw", "tta_raw", "tta_full")])
    pd.DataFrame([{"tile": t, **m} for t, m in D.per_tile(rec, "tta_raw").items()]).to_csv(
        f"{OUT}/per_tile_{TRAIN_EXPERIMENT}.csv", index=False)
    shutil.copy(best, OUT)
else:
    print("part B test skipped:", status)

In [ ]:
# Save
json.dump(RESULTS, open(f"{OUT}/massachusetts_results.json", "w"), indent=1, default=str)
shutil.make_archive("/kaggle/working/massachusetts_results", "zip", OUT)
print("saved /kaggle/working/massachusetts_results.zip")